In [ ]:
# Import Data Manipulation & Visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import NLP libraries
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Import Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Download required NLTK data (This will only download the first time you run it)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# 1. Load the dataset
# Ensure 'combined_data.csv' is in the exact same directory as your Jupyter Notebook/Python file
df = pd.read_csv('combined_data.csv')

# 2. Drop any immediate null values just to be safe
df.dropna(inplace=True)

# 3. Drop exact duplicate emails to prevent model bias
df.drop_duplicates(inplace=True)

# 4. Display basic information to confirm it loaded correctly
print(f"Dataset Shape after dropping nulls and duplicates: {df.shape}")
df.head()

In [ ]:
# 1. Create a new column to store the word count of each email
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

# 2. --- Outlier Handling ---
print(f"Original dataset size: {df.shape[0]} rows")

# We will drop the extreme bottom 1% (too short) and top 1% (absurdly long)
lower_bound = df['word_count'].quantile(0.01)
upper_bound = df['word_count'].quantile(0.99)

# Filter the dataframe to keep only rows within these bounds
df = df[(df['word_count'] >= lower_bound) & (df['word_count'] <= upper_bound)]

print(f"Dataset size after removing length outliers: {df.shape[0]} rows")
print(f"Removed emails with fewer than {int(lower_bound)} or more than {int(upper_bound)} words.\n")

# 3. --- Exploratory Data Analysis (Visualizations) ---
plt.figure(figsize=(14, 5))

# Plot 1: Class Distribution (Spam vs Ham)
# This shows if your dataset is balanced or imbalanced
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='label') 
plt.title('Class Distribution (0: Ham, 1: Spam)', fontsize=14, fontweight='bold')
plt.xlabel('Email Type', fontsize=12)
plt.ylabel('Total Count', fontsize=12)

# Plot 2: Word Count Distribution 
# This shows if spam emails tend to be longer or shorter than normal emails
plt.subplot(1, 2, 2)
sns.histplot(data=df, x='word_count', hue='label', bins=50, kde=True)
plt.title('Word Count Distribution by Class', fontsize=14, fontweight='bold')
plt.xlabel('Number of Words', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# Adjust layout so labels don't overlap, then display
plt.tight_layout()
plt.show()

# Drop the word_count column as we won't need it for model training
df.drop('word_count', axis=1, inplace=True)

In [ ]:
# Initialize the Lemmatizer and Stopwords list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """
    Custom function to clean, tokenize, and lemmatize text data.
    """
    # 1. Convert everything to lowercase
    text = str(text).lower()
    
    # 2. Use Regex to remove anything that is NOT a letter or whitespace
    # This removes numbers, punctuation, and weird characters like "ô¡ô¡"
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 3. Remove the specific placeholder "escapenumber" we saw in the raw data
    text = text.replace('escapenumber', '')
    
    # 4. Tokenize (split into words)
    words = text.split()
    
    # 5. Remove stopwords AND apply Lemmatization
    # We only keep the word if it's not a stopword, and we reduce it to its root form
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    # 6. Join the cleaned words back into a single string
    return " ".join(cleaned_words)

# Show a "Before" example
print("--- Before Cleaning ---")
print(df['text'].iloc[21]) # Grabbing a random row (row 22 from your screenshot)

# Apply the cleaning function to the entire 'text' column
# We create a new column 'cleaned_text' so we don't destroy the original data just yet
print("\nApplying advanced NLP cleaning pipeline. This might take a minute...")
df['cleaned_text'] = df['text'].apply(clean_text)

# Drop any rows that became completely empty after cleaning (e.g., an email that was ONLY punctuation)
df = df[df['cleaned_text'].str.strip() != '']

# Show the "After" example
print("\n--- After Cleaning ---")
print(df['cleaned_text'].iloc[21])

# Now we can safely drop the raw text column
df.drop('text', axis=1, inplace=True)
df.head()

In [ ]:
# 1. Initialize the TF-IDF Vectorizer
# We limit to the top 5,000 words to save memory and prevent overfitting.
# Note: We DO NOT need stop_words='english' here anymore because our custom NLTK function already handled it!
# ngram_range=(1, 2) means look at single words AND 2-word combinations
# We bumped max_features from 5,000 to 15,000 to make room for all the new Bigrams!
tfidf = TfidfVectorizer(ngram_range=(1, 2))
# tfidf = TfidfVectorizer(max_features=5000)

print("Converting text to numerical vectors (TF-IDF)...")
# 2. Fit the vectorizer on our cleaned text and transform it into a matrix (X)
X = tfidf.fit_transform(df['cleaned_text'])

# 3. Define our target variable (y)
y = df['label']

# 4. Split the data: 80% for training the models, 20% for testing them
# Using random_state=42 ensures that you and your friend will get the EXACT same split
# even if you run this on your MSI laptop and he runs it on his.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n--- Data Split Successful ---")
print(f"Training Data: {X_train.shape[0]} emails")
print(f"Testing Data:  {X_test.shape[0]} emails")
print(f"Number of Features (Words): {X_train.shape[1]}")

In [ ]:
from sklearn.utils import resample

# Dictionary to store all our scores so we can graph them later!
model_results = {}

def evaluate_model(model_name, y_true, y_pred):
    """Helper function to print and return evaluation metrics"""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"--- {model_name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f} (How many flagged as spam were actually spam?)")
    print(f"Recall:    {rec:.4f} (How much of the total spam did we catch?)")
    print(f"F1-Score:  {f1:.4f}\n")
    
    return {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}

# ==========================================
# 1. Naive Bayes (Standard for Text)
# ==========================================
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
model_results['Naive Bayes'] = evaluate_model('Naive Bayes', y_test, nb_pred)

# ==========================================
# 2. Logistic Regression
# ==========================================
lr_model = LogisticRegression(C=0.5, max_iter=1000) # Increased max_iter to ensure it converges
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
model_results['Logistic Regression'] = evaluate_model('Logistic Regression', y_test, lr_pred)

# ==========================================
# 3. K-Nearest Neighbors (Sampled)
# ==========================================
print("Sampling training data for KNN to 10,000 rows to prevent memory overload...")
# We only sample the TRAINING data. The test data remains exactly the same for a fair comparison.
X_train_knn, y_train_knn = resample(X_train, y_train, n_samples=10000, random_state=42)

# Using k=5 as a solid baseline. 
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_knn, y_train_knn)
knn_pred = knn_model.predict(X_test)
model_results['KNN (Sampled 10k)'] = evaluate_model('K-Nearest Neighbors', y_test, knn_pred)

In [ ]:
import joblib

# 1. --- Prepare Data for Plotting ---
# Convert our dictionary of results into a pandas DataFrame
results_df = pd.DataFrame(model_results).T

# 2. --- Visualize the Metrics ---
# We use a grouped bar chart to compare all 4 metrics across all 3 models
ax = results_df.plot(kind='bar', figsize=(14, 7), colormap='viridis', edgecolor='black', width=0.8)

plt.title('Final Model Performance Comparison (CA-2 Project)', fontsize=16, fontweight='bold')
plt.xlabel('Machine Learning Models', fontsize=12)
plt.ylabel('Score (0.0 to 1.0)', fontsize=12)
plt.xticks(rotation=0) # Keep the model names horizontal
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=4, title='Metrics')

plt.ylim(0.0, 1.05) 
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 3. --- Add Score Labels ---
# This loop adds the exact decimal score on top of every single bar
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10)

plt.tight_layout()
plt.show()

# ==========================================
# 4. Save the Winning Model & Vectorizer
# ==========================================
print("\n--- Saving Assets for Streamlit Deployment ---")

# We MUST save both! 
# The Streamlit app needs the exact same vectorizer to convert new, real-world 
# emails into the same 5000-dimension vector format before the model can read it.
joblib.dump(lr_model, 'spam_model_lr.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print("Success! 'spam_model_lr.pkl' and 'tfidf_vectorizer.pkl' are saved in your folder.")